In [1]:
import os

import pandas as pd

from framework.dataset_specification import NamedDatasetSpecifications, DatasetSpecification
from framework.enumerations import EvaluationDatasetSampling
from framework.flow_transformer_multi_classification import FlowTransformer
from framework.flow_transformer_parameters import FlowTransformerParameters
from implementations.classification_heads import *
from implementations.input_encodings import *
from implementations.pre_processings import StandardPreProcessing
from implementations.transformers.basic_transformers import BasicTransformer
from implementations.transformers.named_transformers import *

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
encodings = [
    NoInputEncoder(),
    RecordLevelEmbed(64),
    CategoricalFeatureEmbed(EmbedLayerType.Dense, 16),
    CategoricalFeatureEmbed(EmbedLayerType.Lookup, 16),
    CategoricalFeatureEmbed(EmbedLayerType.Projection, 16),
    RecordLevelEmbed(64, project=True)
]

classification_heads = [
    LastTokenClassificationHead(),
    FlattenClassificationHead(),
    GlobalAveragePoolingClassificationHead(),
    CLSTokenClassificationHead(),
    FeaturewiseEmbedding(project=False),
    FeaturewiseEmbedding(project=True),
]

transformers = [
    BasicTransformer(2, 128, n_heads=2),
    BasicTransformer(2, 128, n_heads=2, is_decoder=True),
    GPTSmallTransformer(),
    BERTSmallTransformer()
]

In [ ]:
custom_dataset_spec = DatasetSpecification(
    include_fields=['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'FLOW_DURATION_MILLISECONDS', 'OUT_PKTS', 'OUT_BYTES', 'IN_PKTS', 'IN_BYTES', 'L7_PROTO', 'TCP_FLAGS'],
    categorical_fields=['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'TCP_FLAGS'],
    class_column="Attack",
    benign_label="Benign"
)

datasets = [
    ("Balanced_Dataset_Multi", "../FlowTransformer/data/balanced_dataset_tcp_udp.csv", custom_dataset_spec, 0.2, EvaluationDatasetSampling.RandomRows)
]

In [ ]:
pre_processing = StandardPreProcessing(n_categorical_levels=32)

# Define the transformer
ft = FlowTransformer(pre_processing=pre_processing,
                     input_encoding=encodings[0],
                     sequential_model=transformers[0],
                     classification_head=classification_heads[0],
                     params=FlowTransformerParameters(window_size=8, mlp_layer_sizes=[128], mlp_dropout=0.1))

# Load the specific dataset
dataset_name, dataset_path, dataset_specification, eval_percent, eval_method = datasets[0]
ft.load_dataset(dataset_name, dataset_path, dataset_specification, evaluation_dataset_sampling=eval_method, evaluation_percent=eval_percent)


In [ ]:
m = ft.build_model()
m.summary()


In [ ]:
import tensorflow as tf



# 关闭 jit_compile 以避免某些环境下图导出不稳定；并启用梯度裁剪减少训练发散风险

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)

m.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'], jit_compile=False)



# Get the evaluation results

eval_results: pd.DataFrame

(train_results, eval_results, final_epoch) = ft.evaluate(m, batch_size=128, epochs=5, steps_per_epoch=64, early_stopping_patience=5)



# print(eval_results)


In [ ]:
# 仅导出 ONNX + 元信息（RKNN 转换放到独立脚本 tools/onnx_to_rknn.py 在另一台机器执行）

print("\n--> 1. 正在将 Keras 模型导出为 ONNX 格式...")



import os

import json

import numpy as np

import tf2onnx

import tensorflow as tf



onnx_save_path = os.path.join("..", "models", "t_multi_class.onnx")

rknn_save_path = os.path.join("..", "models", "transformer_multi_class_model.rknn")

meta_save_path = os.path.splitext(onnx_save_path)[0] + ".meta.json"



# 导出模型输入元信息，供后续 RKNN 推理/主程序读取

model_input_names = []

model_expected_dims = {}

for tensor in m.inputs:

    name = tensor.name.split(':')[0] if ':' in tensor.name else tensor.name

    model_input_names.append(name)

    shape = tensor.shape.as_list() if hasattr(tensor.shape, "as_list") else list(tensor.shape)

    if len(shape) == 3 and shape[-1] is not None and int(shape[-1]) != 1:

        feat_name = name.replace("input_", "", 1)

        model_expected_dims[feat_name] = int(shape[-1])



model_meta = {

    "model_input_names": model_input_names,

    "model_expected_dims": model_expected_dims,

    "model_output_name": m.outputs[0].name.split(':')[0] if m.outputs else None

}

with open(meta_save_path, "w", encoding="utf-8") as f:

    json.dump(model_meta, f, ensure_ascii=False, indent=2)

print(f"模型元信息已保存至 {meta_save_path}")



# 固定 batch 维度，避免导出动态维导致后续工具链不兼容

input_signature = []

for tensor in m.inputs:

    shape = list(tensor.shape)

    shape[0] = 1

    name = tensor.name.split(':')[0] if ':' in tensor.name else tensor.name

    input_signature.append(tf.TensorSpec(shape=tuple(shape), dtype=tensor.dtype, name=name))



model_proto, _ = tf2onnx.convert.from_keras(

    m,

    input_signature=input_signature,

    output_path=onnx_save_path,

    opset=13,

)

print(f"ONNX 模型已保存至 {onnx_save_path}")



# 导出后检查 ONNX 是否有 NaN/Inf 权重，避免后续 RKNN 出现恒定输出

print("\n--> 1.1 正在校验 ONNX 权重...")

import onnx

from onnx import numpy_helper



onnx_model = onnx.load(onnx_save_path)

bad_inits = []

for init in onnx_model.graph.initializer:

    arr = numpy_helper.to_array(init)

    if np.issubdtype(arr.dtype, np.floating):

        if np.isnan(arr).any() or np.isinf(arr).any():

            bad_inits.append(init.name)



if bad_inits:

    raise RuntimeError(

        "导出的 ONNX 含 NaN/Inf 权重，已停止。"

        f"\n异常参数数量: {len(bad_inits)}，示例: {bad_inits[:5]}"

    )



print("ONNX 权重校验通过")



print("\n--> 2. 本机到此结束（仅训练+导出 ONNX）")

print("请把以下文件拷贝到转换机：")

print(f"  - {onnx_save_path}")

print(f"  - {meta_save_path}")



print("\n在转换机执行：")

print(

    "python tools/onnx_to_rknn.py "

    f"--onnx {onnx_save_path} "

    f"--output {rknn_save_path} "

    "--target rk3588 --copy-meta"

)
